# Logistics Knowledge Assistant - RAG with Databricks Native Stack

## Architecture Overview

This notebook builds a **Retrieval-Augmented Generation (RAG)** application using only Databricks-native features:

* **Unity Catalog** for data governance and organization
* **Volumes** for PDF document storage
* **Delta Tables** for structured chunk storage
* **Databricks Vector Search** with Delta Sync for semantic search (managed embeddings)
* **Databricks Foundation Model APIs** for embeddings and LLM inference

## What You'll Build

1. **Data Layer**: UC catalog/schema/volume → PDF ingestion → chunked Delta table
2. **Index Layer**: Vector Search endpoint → Delta Sync index with auto-embeddings
3. **Application Layer**: Retrieval function + grounded generation with citations
4. **Testing**: End-to-end examples

## Prerequisites

* Unity Catalog enabled workspace
* PDFs uploaded to a Volume (we'll create one)
* Access to Foundation Model APIs (check your **Serving** tab for available endpoints)

---

**Each section below is independently runnable.** Execute them in order the first time, then you can re-run individual sections as needed.

## Section 1: Unity Catalog Setup

### What This Does

Creates the foundational Unity Catalog structure:

* **Catalog**: Top-level namespace for organizing data
* **Schema**: Logical grouping within the catalog
* **Volume**: Managed cloud storage for files (PDFs, images, etc.)

### Why This Structure

* **Governance**: UC provides centralized access control, lineage, and auditing
* **Volumes**: Native file storage that's UC-governed (better than raw cloud storage paths)
* **Isolation**: Keeps your RAG application's data separate from other projects

Volumes are ideal for unstructured data (PDFs) while Delta tables handle structured data (chunks, embeddings).

In [0]:
# CONFIGURATION: Adjust these names for your workspace
CATALOG_NAME = "logistics_rag"
SCHEMA_NAME = "knowledge_base"
VOLUME_NAME = "pdf_documents"

# Full paths we'll use throughout
FULL_SCHEMA = f"{CATALOG_NAME}.{SCHEMA_NAME}"
VOLUME_PATH = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{VOLUME_NAME}"

print(f"Setting up Unity Catalog structure:")
print(f"  Catalog: {CATALOG_NAME}")
print(f"  Schema: {FULL_SCHEMA}")
print(f"  Volume: {VOLUME_PATH}")

# Create catalog (if it doesn't exist)
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")
print(f"✓ Catalog '{CATALOG_NAME}' ready")

# Create schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FULL_SCHEMA}")
print(f"✓ Schema '{FULL_SCHEMA}' ready")

# Create volume for PDF storage
spark.sql(f"CREATE VOLUME IF NOT EXISTS {FULL_SCHEMA}.{VOLUME_NAME}")
print(f"✓ Volume created at: {VOLUME_PATH}")
print(f"\n📁 Upload your PDF files to: {VOLUME_PATH}")
print(f"   (Use the Catalog Explorer UI or dbutils.fs.cp)")

Setting up Unity Catalog structure:
  Catalog: logistics_rag
  Schema: logistics_rag.knowledge_base
  Volume: /Volumes/logistics_rag/knowledge_base/pdf_documents
✓ Catalog 'logistics_rag' ready
✓ Schema 'logistics_rag.knowledge_base' ready
✓ Volume created at: /Volumes/logistics_rag/knowledge_base/pdf_documents

📁 Upload your PDF files to: /Volumes/logistics_rag/knowledge_base/pdf_documents
   (Use the Catalog Explorer UI or dbutils.fs.cp)


## Section 2: PDF Loading, Text Extraction, and Chunking

### What This Does

1. **Loads PDFs** from the Volume using PyMuPDF (pre-installed in Databricks)
2. **Extracts text** page-by-page
3. **Chunks text** into ~800-1000 tokens with 120-token overlap
4. **Writes to Delta table** with columns: `id`, `chunk_text`, `source_doc`, `page`

### Chunking Strategy Explained

* **800-1000 tokens**: Balances context richness vs. retrieval precision
  * Too small → fragments lose meaning
  * Too large → irrelevant text dilutes relevance scores
* **120-token overlap**: Prevents information loss at chunk boundaries
  * Example: A key fact split across chunks stays retrievable
* **Token-based** (not character-based): LLMs process tokens, so we size chunks in their native units

### Schema Design

* `id`: Unique identifier for each chunk
* `chunk_text`: The actual text content (what gets embedded)
* `source_doc`: Original PDF filename (for citations)
* `page`: Page number (for precise references)

In [0]:
%pip install pymupdf tiktoken --quiet
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import fitz  # PyMuPDF
import tiktoken
import os
from typing import List, Dict
import hashlib

# Use cl100k_base encoding (compatible with most modern LLMs)
encoder = tiktoken.get_encoding("cl100k_base")

def chunk_text(text: str, chunk_size: int = 900, overlap: int = 120) -> List[str]:
    """
    Chunk text into token-sized pieces with overlap.
    
    Args:
        text: Input text to chunk
        chunk_size: Target tokens per chunk (default 900 for ~800-1000 range)
        overlap: Overlapping tokens between chunks (prevents boundary loss)
    
    Returns:
        List of text chunks
    """
    tokens = encoder.encode(text)
    chunks = []
    
    start = 0
    while start < len(tokens):
        # Extract chunk
        end = start + chunk_size
        chunk_tokens = tokens[start:end]
        chunk_text = encoder.decode(chunk_tokens)
        chunks.append(chunk_text)
        
        # Move start position (with overlap)
        start = end - overlap
        
        # Prevent infinite loop on short texts
        if end >= len(tokens):
            break
    
    return chunks

def extract_pdf_chunks(pdf_path: str) -> List[Dict]:
    """
    Extract and chunk text from a PDF file.
    
    Returns:
        List of dicts with keys: chunk_text, source_doc, page
    """
    doc = fitz.open(pdf_path)
    filename = os.path.basename(pdf_path)
    all_chunks = []
    
    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text()
        
        # Skip empty pages
        if not text.strip():
            continue
        
        # Chunk the page text
        chunks = chunk_text(text)
        
        for chunk in chunks:
            all_chunks.append({
                "chunk_text": chunk,
                "source_doc": filename,
                "page": page_num + 1  # 1-indexed for human readability
            })
    
    doc.close()
    return all_chunks

print("✓ PDF processing functions loaded")
print(f"  Chunking: ~900 tokens per chunk, 120 token overlap")
print(f"  Tokenizer: cl100k_base (OpenAI-compatible)")

✓ PDF processing functions loaded
  Chunking: ~900 tokens per chunk, 120 token overlap
  Tokenizer: cl100k_base (OpenAI-compatible)


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import uuid

# Table configuration
TABLE_NAME = f"{FULL_SCHEMA}.logistics_chunks"

print(f"Processing PDFs from: {VOLUME_PATH}")

# List all PDFs in the volume
pdf_files = [f.path for f in dbutils.fs.ls(VOLUME_PATH) if f.path.endswith('.pdf')]

if not pdf_files:
    print(f"⚠️  No PDF files found in {VOLUME_PATH}")
    print(f"   Please upload PDFs to the volume before running this cell.")
else:
    print(f"Found {len(pdf_files)} PDF file(s)")
    
    # Process all PDFs
    all_chunks = []
    for pdf_path in pdf_files:
        print(f"  Processing: {os.path.basename(pdf_path)}")
        chunks = extract_pdf_chunks(pdf_path.replace('dbfs:', ''))
        all_chunks.extend(chunks)
    
    print(f"\n✓ Extracted {len(all_chunks)} total chunks")
    
    # Add unique IDs
    for i, chunk in enumerate(all_chunks):
        # Create deterministic ID from content hash + index
        content_hash = hashlib.md5(chunk['chunk_text'].encode()).hexdigest()[:8]
        chunk['id'] = f"{content_hash}_{i}"
    
    # Create DataFrame
    schema = StructType([
        StructField("id", StringType(), False),
        StructField("chunk_text", StringType(), False),
        StructField("source_doc", StringType(), False),
        StructField("page", IntegerType(), False)
    ])
    
    # Reorder to match schema
    rows = [(c['id'], c['chunk_text'], c['source_doc'], c['page']) for c in all_chunks]
    chunks_df = spark.createDataFrame(rows, schema)
    
    # Write to Delta table
    chunks_df.write.format("delta").mode("overwrite").saveAsTable(TABLE_NAME)
    
    print(f"\n✓ Created Delta table: {TABLE_NAME}")
    print(f"  Columns: id, chunk_text, source_doc, page")
    print(f"  Total chunks: {len(all_chunks)}")
    
    # Show sample
    display(spark.table(TABLE_NAME).limit(3))

Processing PDFs from: /Volumes/logistics_rag/knowledge_base/pdf_documents
Found 16 PDF file(s)
  Processing: FedEx_Freight_Shipping_Guide.pdf
  Processing: GS1_Global_Traceability_Standard.pdf
  Processing: GS1_Logistic_Label_Guideline.pdf
  Processing: GS1_Traceability_Checklist_Guide.pdf
  Processing: UPS_Packaging_Guidelines.pdf
  Processing: arXiv_AI_Sustainable_Logistics_Optimization.pdf
  Processing: arXiv_Demand_Forecasting_Supply_Chain.pdf
  Processing: arXiv_Inventory_Management_ML.pdf
  Processing: arXiv_LLM_Agents_Supply_Chain.pdf
  Processing: arXiv_LLM_Network_Optimization_Supply_Chain.pdf
  Processing: arXiv_Last_Mile_Delivery_Optimization.pdf
  Processing: arXiv_ML_Supply_Chain_Optimization.pdf
  Processing: arXiv_OptiRepair_LLM_Supply_Chain.pdf
  Processing: arXiv_Supply_Chain_Generative_Simulation.pdf
  Processing: arXiv_Supply_Chain_Resilience_LLM.pdf
  Processing: arXiv_Warehouse_Automation_Robotics.pdf

✓ Extracted 468 total chunks

✓ Created Delta table: logistics_

id,chunk_text,source_doc,page
d9095171_0,"Freight Shipping Guide FedEx is dedicated to taking care of you and your customers by providing fast, reliable, intact shipments. Customer Service 1.866.393.4585",FedEx_Freight_Shipping_Guide.pdf,1
58611d36_1,"- 1 - SHIPMENT CLASSIFICATION Four freight characteristics are used to determine shipment classification: l Density - how heavy the shipment is for the space it occupies–expressed in pounds per cubic feet (PCF). l Stowability – how well the shipment uses trailer space considering: l Excess size or weight. l If the shipment can be stacked on with other freight. l Loading restrictions due to government regulations. l Ease of handling - special care may be required due to size, weight or configuration. l Liability - measured by such things as: l Value per pound. l Susceptibility to theft or damage. l Liability to damage. l Perishability. l Tendency to damage other freight, spontaneously combust or explode. For additional assistance, contact your sales executive. Additional information is available from the NMFTA at 1.866.411.NMFC (6632) or go to www.nmfta.org. DENSITY To secure the best possible LTL (less-than- truckload) shipping rates, always package your products using the least amount of space, and provide accurate weight(s). This increases shipment density–a plus for you as a shipper. l Density = Weight (lbs.)/Cubic Feet l Cubic feet = the greatest dimensions of the handling unit divided by 1728 - (L” x W” x H”)/1728 l Cubic feet for round items such as cylinders, drums, etc. = (H” x D” x D”)/1728 where D = diameter Add all handling unit weights together and divide by the total number of cubic feet for all handling units. - 2 - L” H” W” - 4 - PACKAGING CONSIDERATIONS Layers of shrinkwrap must overlap, especially with tall pallets. Wrap upward, overlapping film by 50 percent. Result: Stabilizes the freight and keeps it from tipping over. Twist shrinkwrap every other time around freight. Result: Increases strength of the wrap. Tie shrinkwrap to pallet before wrapping, and wrap all freight securely to the pallet. Result: Limits chances of freight sliding off pallet or falling off pallet. Only use pallets that are in good condition. Result: Avoids damage to freight during transit caused by boards caving in or wood and nails damaging freight. Treated hardwood pallets are required for international destinations. - 3 - BILL OF LADING The Bill of Lading (BOL) is a required legal contract that verifies that the carrier has received the shipment as described and is obligated to deliver that shipment in good condition to the consignee (person receiving the freight). Key information to include on the BOL: l Description of what is being shipped:* l Clearly describe all commodities. l List NMFC item number(s) and class(es). l State accurate shipment weight(s). l Provide a piece count and number of handling units. l Where is the shipment’s ultimate destination? l List any special handling requirements. l Who will pay the shipping charges: l Shipper = prepaid [PPD]. l Consignee = collect [COL]. l Third party. l For international shipments, include the name and phone number of the customs broker. * An accurate commodity description, including the correct NMFC item number, freight class and weight, is required to avoid billing issues and promotes invoicing accuracy.",FedEx_Freight_Shipping_Guide.pdf,2
3ee2afeb_2,"DO NOT place any freight on a pallet without securing the freight to the pallet. Result: Freight is always in constant motion/vibration during transit. Anything not secured to pallet will fall from pallet. DO NOT load bags directly on the pallet. Result: Bags are easily torn. Always place cardboard between the pallet and around the sides of the pallet whenever possible. - 6 - - 5 - Place cardboard or chipboard between freight and pallet. Result: Reduces freight damage from forks and wood or nails working into freight. Stack smaller, lighter items on heavier items. Result: Reduce possibl

## Section 3: Vector Search Endpoint and Delta Sync Index

### What Is Delta Sync?

**Delta Sync** is Databricks' automatic synchronization between a Delta table and a Vector Search index:

* **Managed Embeddings**: Databricks generates embeddings using a Foundation Model (you don't manage the embedding logic)
* **Auto-Sync**: When the source Delta table updates, the index automatically refreshes
* **Serverless**: No infrastructure to manage

### How It Works

1. **Vector Search Endpoint**: Compute infrastructure for serving the index (think of it like a SQL warehouse for vectors)
2. **Delta Sync Index**: Watches your Delta table, embeds the `chunk_text` column, stores vectors + metadata
3. **Querying**: You pass a question → endpoint embeds it → returns top-k similar chunks

### Why This Matters

* **No ETL**: You write to the Delta table; embeddings happen automatically
* **Always Current**: New documents → append to table → index updates automatically
* **Governance**: Inherits UC permissions from the source table

In [0]:
%pip install databricks-vectorsearch --quiet
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from databricks.vector_search.client import VectorSearchClient
import time

# CONFIGURATION: Re-declare after Python restart
CATALOG_NAME = "logistics_rag"
SCHEMA_NAME = "knowledge_base"
FULL_SCHEMA = f"{CATALOG_NAME}.{SCHEMA_NAME}"
TABLE_NAME = f"{FULL_SCHEMA}.logistics_chunks"

# Initialize client
vsc = VectorSearchClient(disable_notice=True)

# CONFIGURATION: Vector Search setup
ENDPOINT_NAME = "logistics_rag_endpoint"
INDEX_NAME = f"{FULL_SCHEMA}.logistics_chunks_index"
EMBEDDING_ENDPOINT = "databricks-bge-large-en"

print(f"Setting up Vector Search infrastructure...\n")

# Step 0: Enable Change Data Feed (required for Delta Sync)
print(f"[1/3] Enabling Change Data Feed on {TABLE_NAME}...")
spark.sql(f"ALTER TABLE {TABLE_NAME} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
print(f"      ✓ Change Data Feed enabled\n")

# Step 1: Create/verify endpoint
print(f"[2/3] Setting up Vector Search endpoint '{ENDPOINT_NAME}'...")
try:
    endpoint_info = vsc.get_endpoint(ENDPOINT_NAME)
    endpoint_status = endpoint_info.get("endpoint_status", {}).get("state")
    
    if endpoint_status == "ONLINE":
        print(f"      ✓ Endpoint already exists and is ONLINE")
    else:
        print(f"      Endpoint exists but status is: {endpoint_status}")
        print(f"      Waiting for endpoint to be ready (timeout: 15 min)...")
        
        max_wait = 900  # 15 minutes
        waited = 0
        while waited < max_wait:
            status = vsc.get_endpoint(ENDPOINT_NAME).get("endpoint_status", {}).get("state")
            if status == "ONLINE":
                print(f"      ✓ Endpoint is now ONLINE")
                break
            print(f"      Status: {status}... waited {waited}s")
            time.sleep(30)
            waited += 30
        else:
            raise TimeoutError(f"Endpoint did not come online after {max_wait}s")
            
except Exception as e:
    if "does not exist" in str(e).lower() or "RESOURCE_DOES_NOT_EXIST" in str(e):
        print(f"      Creating new endpoint (this takes 5-10 min)...")
        vsc.create_endpoint(
            name=ENDPOINT_NAME,
            endpoint_type="STANDARD"
        )
        print(f"      Waiting for provisioning (timeout: 15 min)...")
        
        max_wait = 900
        waited = 0
        while waited < max_wait:
            status = vsc.get_endpoint(ENDPOINT_NAME).get("endpoint_status", {}).get("state")
            if status == "ONLINE":
                print(f"      ✓ Endpoint is ONLINE")
                break
            print(f"      Status: {status}... waited {waited}s")
            time.sleep(30)
            waited += 30
        else:
            raise TimeoutError(f"Endpoint did not come online after {max_wait}s")
    else:
        raise

# Step 2: Create/verify index
print(f"\n[3/3] Setting up Delta Sync index '{INDEX_NAME}'...")
try:
    index = vsc.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)
    index_details = index.describe()
    index_status = index_details.get("status", {})
    detailed_state = index_status.get("detailed_state", "UNKNOWN")
    indexed_rows = index_status.get("indexed_row_count", 0)
    is_ready = index_status.get("ready", False)
    
    # Check if index is in a ready state
    ready_states = [
        "ONLINE",
        "ONLINE_AVAILABLE", 
        "ONLINE_NO_PENDING_UPDATE",
        "ONLINE_TRIGGERED_UPDATE"
    ]
    
    if is_ready or detailed_state in ready_states:
        print(f"      ✓ Index already exists and is ready")
        print(f"      Status: {detailed_state}")
        print(f"      Indexed rows: {indexed_rows}")
    else:
        print(f"      Index exists but not ready yet: {detailed_state}")
        print(f"      Waiting for index sync (timeout: 20 min)...")
        
        max_wait = 1200  # 20 minutes
        waited = 0
        while waited < max_wait:
            index = vsc.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)
            index_details = index.describe()
            status_obj = index_details.get("status", {})
            detailed_state = status_obj.get("detailed_state", "UNKNOWN")
            is_ready = status_obj.get("ready", False)
            
            if is_ready or detailed_state in ready_states:
                indexed_rows = status_obj.get("indexed_row_count", 0)
                print(f"      ✓ Index is ready: {detailed_state}")
                print(f"      Indexed rows: {indexed_rows}")
                break
            
            print(f"      Status: {detailed_state}... waited {waited}s")
            time.sleep(30)
            waited += 30
        else:
            print(f"      ⚠️  Timeout after {max_wait}s, but index may still be syncing")
            print(f"      Current status: {detailed_state}")
            print(f"      Check the UI for progress")
        
except Exception as e:
    if "does not exist" in str(e).lower() or "INDEX_DOES_NOT_EXIST" in str(e):
        print(f"      Creating new index...")
        vsc.create_delta_sync_index(
            endpoint_name=ENDPOINT_NAME,
            index_name=INDEX_NAME,
            source_table_name=TABLE_NAME,
            pipeline_type="TRIGGERED",
            primary_key="id",
            embedding_source_column="chunk_text",
            embedding_model_endpoint_name=EMBEDDING_ENDPOINT
        )
        print(f"      Index creation initiated")
        print(f"      Waiting for initial sync (timeout: 20 min)...")
        
        max_wait = 1200
        waited = 0
        ready_states = ["ONLINE", "ONLINE_AVAILABLE", "ONLINE_NO_PENDING_UPDATE", "ONLINE_TRIGGERED_UPDATE"]
        
        while waited < max_wait:
            try:
                index = vsc.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)
                index_details = index.describe()
                status_obj = index_details.get("status", {})
                detailed_state = status_obj.get("detailed_state", "UNKNOWN")
                is_ready = status_obj.get("ready", False)
                
                if is_ready or detailed_state in ready_states:
                    indexed_rows = status_obj.get("indexed_row_count", 0)
                    print(f"      ✓ Index is ready: {detailed_state}")
                    print(f"      Indexed rows: {indexed_rows}")
                    break
                
                print(f"      Status: {detailed_state}... waited {waited}s")
            except Exception as check_error:
                print(f"      Provisioning... waited {waited}s")
            
            time.sleep(30)
            waited += 30
        else:
            print(f"      ⚠️  Timeout after {max_wait}s, but index may still be syncing")
            print(f"      Check the UI or re-run this cell later")
    else:
        raise

print(f"\n🎉 Vector Search setup complete!")
print(f"   Endpoint: {ENDPOINT_NAME}")
print(f"   Index: {INDEX_NAME}")
print(f"   Embedding model: {EMBEDDING_ENDPOINT}")
print(f"\n💡 Tip: If setup timed out, check status in UI:")
print(f"   Catalog > {CATALOG_NAME} > {SCHEMA_NAME} > {INDEX_NAME.split('.')[-1]}")

Setting up Vector Search infrastructure...

[1/3] Enabling Change Data Feed on logistics_rag.knowledge_base.logistics_chunks...
      ✓ Change Data Feed enabled

[2/3] Setting up Vector Search endpoint 'logistics_rag_endpoint'...
      ✓ Endpoint already exists and is ONLINE

[3/3] Setting up Delta Sync index 'logistics_rag.knowledge_base.logistics_chunks_index'...
      ✓ Index already exists and is ready
      Status: ONLINE_NO_PENDING_UPDATE
      Indexed rows: 468

🎉 Vector Search setup complete!
   Endpoint: logistics_rag_endpoint
   Index: logistics_rag.knowledge_base.logistics_chunks_index
   Embedding model: databricks-bge-large-en

💡 Tip: If setup timed out, check status in UI:
   Catalog > logistics_rag > knowledge_base > logistics_chunks_index


## Section 4: Retrieval Function

### What This Does

Implements **semantic search**: given a natural language question, return the most relevant document chunks.

### How It Works

1. **Query Embedding**: Question → embedding model → vector
2. **Similarity Search**: Compare query vector against all chunk vectors
3. **Top-K Retrieval**: Return the k most similar chunks (we use k=5)

### Why k=5?

* **Balance**: Enough context for the LLM without overwhelming it
* **Token Limit**: 5 chunks × ~900 tokens = ~4500 tokens (well within most LLM context windows)
* **Precision vs. Recall**: More chunks = more context but potentially more noise

You can adjust `k` based on your needs:
* k=3: Faster, more precise (good for narrow questions)
* k=10: More comprehensive (good for broad topics)

In [0]:
from typing import List, Dict

def retrieve_chunks(question: str, k: int = 5) -> List[Dict]:
    """
    Retrieve the top-k most relevant chunks for a question.
    
    Args:
        question: Natural language question
        k: Number of chunks to retrieve (default 5)
    
    Returns:
        List of dicts with keys: chunk_text, source_doc, page, score
    """
    # Search the index
    results = vsc.get_index(
        endpoint_name=ENDPOINT_NAME,
        index_name=INDEX_NAME
    ).similarity_search(
        query_text=question,
        columns=["id", "chunk_text", "source_doc", "page"],
        num_results=k
    )
    
    # Extract results
    chunks = []
    if results and 'result' in results and 'data_array' in results['result']:
        for row in results['result']['data_array']:
            chunks.append({
                "id": row[0],
                "chunk_text": row[1],
                "source_doc": row[2],
                "page": row[3],
                "score": row[4] if len(row) > 4 else None  # Similarity score
            })
    
    return chunks

# Test the retrieval function
test_question = "What are the shipping lead times for international orders?"
print(f"Test Query: '{test_question}'\n")

test_results = retrieve_chunks(test_question, k=3)
print(f"Retrieved {len(test_results)} chunks:\n")

for i, chunk in enumerate(test_results, 1):
    print(f"[Chunk {i}] {chunk['source_doc']} (Page {chunk['page']})")
    print(f"  Score: {chunk['score']:.4f}" if chunk['score'] else "")
    print(f"  Text: {chunk['chunk_text'][:150]}...\n")

Test Query: 'What are the shipping lead times for international orders?'

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Retrieved 3 chunks:

[Chunk 1] FedEx_Freight_Shipping_Guide.pdf (Page 4.0)
  Score: 0.5564
  Text: - 11 -
MK551/507-FXF/FXNL
28302FF
CAPACITY AND VOLUME 
l When shipping four pallet positions or 5,000 lbs.
	
or more, you may be eligible to receive a...

[Chunk 2] UPS_Packaging_Guidelines.pdf (Page 6.0)
  Score: 0.5533
  Text: 6 
Please note: UPS does not make any warranties, express or implied, regarding this information. It is the responsibility of the Shipper to ensure th...

[Chunk 3] UPS_Packaging_Guidelines.pdf (Page 5.0)
  Score: 0.5496
  Text: 5 
Please note: UPS does not make any warranties, express or implied, regarding this information. It is the responsibility of the Shipper to ensure th...



## Section 5: Grounded Generation Function

### What This Does

Combines retrieval + generation:

1. **Retrieve** relevant chunks (Section 4)
2. **Build prompt** with strict grounding instructions
3. **Generate answer** using Databricks Foundation Model API
4. **Enforce citations** and honesty ("I don't know" when context doesn't help)

### Prompt Engineering Strategy

* **Strict Grounding**: Answer ONLY from provided context (prevents hallucination)
* **Citation Requirement**: Force model to reference which chunks it used
* **Honesty Directive**: Explicit instruction to say "I don't have that information" when context is insufficient

### Why This Matters

* **Trust**: Users know answers are sourced from their documents
* **Auditability**: Citations enable verification
* **Safety**: Reduces hallucination risk in enterprise settings

In [0]:
import requests
import json

# ⚠️ PLACEHOLDER: Confirm the LLM endpoint name from your Serving tab
# Common names: "databricks-meta-llama-3-70b-instruct", "databricks-mixtral-8x7b-instruct"
# Go to: Databricks UI → Serving → Foundation Model APIs
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"  # ← Meta Llama 3.3 70B

def generate_grounded_answer(question: str, k: int = 5) -> Dict:
    """
    Generate an answer grounded in retrieved document chunks.
    
    Args:
        question: User's question
        k: Number of chunks to retrieve
    
    Returns:
        Dict with keys: answer, chunks_used, sources
    """
    # Step 1: Retrieve relevant chunks
    chunks = retrieve_chunks(question, k=k)
    
    if not chunks:
        return {
            "answer": "I don't have any relevant information to answer this question.",
            "chunks_used": [],
            "sources": []
        }
    
    # Step 2: Build context from chunks
    context_parts = []
    for i, chunk in enumerate(chunks, 1):
        context_parts.append(
            f"[Chunk {i}] (Source: {chunk['source_doc']}, Page {chunk['page']})\n"
            f"{chunk['chunk_text']}\n"
        )
    context = "\n".join(context_parts)
    
    # Step 3: Build prompt with strict grounding instructions
    prompt = f"""You are a logistics knowledge assistant. Answer the question using ONLY the information provided in the context below.

IMPORTANT RULES:
1. Answer ONLY based on the context provided. Do not use external knowledge.
2. Cite which chunk(s) you used (e.g., "According to Chunk 1...").
3. If the context doesn't contain the answer, respond: "I don't have that information in the available documents."
4. Be concise but complete.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""
    
    # Step 4: Call Foundation Model API
    # Get workspace token and URL
    token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
    workspace_url = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
    
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }
    
    data = {
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "max_tokens": 500,
        "temperature": 0.1  # Low temperature for more factual responses
    }
    
    response = requests.post(
        f"{workspace_url}/serving-endpoints/{LLM_ENDPOINT}/invocations",
        headers=headers,
        json=data
    )
    
    if response.status_code != 200:
        return {
            "answer": f"Error calling LLM: {response.text}",
            "chunks_used": chunks,
            "sources": []
        }
    
    result = response.json()
    answer = result['choices'][0]['message']['content']
    
    # Step 5: Extract unique sources
    sources = list(set([(c['source_doc'], c['page']) for c in chunks]))
    
    return {
        "answer": answer,
        "chunks_used": chunks,
        "sources": sources
    }

print(f"✓ Grounded generation function loaded")
print(f"  LLM Endpoint: {LLM_ENDPOINT}")
print(f"  Strategy: Strict grounding + citations + honesty")

✓ Grounded generation function loaded
  LLM Endpoint: databricks-meta-llama-3-3-70b-instruct
  Strategy: Strict grounding + citations + honesty


## Section 6: End-to-End Testing

### Test Strategy

We'll test three scenarios:

1. **Factual Question**: Question that should have a clear answer in the docs
2. **Complex Question**: Requires synthesizing information from multiple chunks
3. **Out-of-Scope Question**: Topic not covered in the documents (tests "I don't know" behavior)

Adjust these questions based on your actual PDF content!

In [0]:
# Customize this question based on your PDF content
question1 = "What are the shipping lead times for international orders?"

print(f"🔍 QUESTION: {question1}\n")
print("=" * 80)

result1 = generate_grounded_answer(question1)

print(f"\n📝 ANSWER:\n{result1['answer']}\n")
print("=" * 80)
print(f"\n📚 SOURCES:")
for doc, page in result1['sources']:
    print(f"  • {doc} (Page {page})")

print(f"\n🔢 Retrieved {len(result1['chunks_used'])} chunks")

🔍 QUESTION: What are the shipping lead times for international orders?

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

📝 ANSWER:
I don't have that information in the available documents.


📚 SOURCES:
  • FedEx_Freight_Shipping_Guide.pdf (Page 4.0)
  • UPS_Packaging_Guidelines.pdf (Page 6.0)
  • GS1_Global_Traceability_Standard.pdf (Page 39.0)
  • UPS_Packaging_Guidelines.pdf (Page 5.0)
  • arXiv_Supply_Chain_Generative_Simulation.pdf (Page 7.0)

🔢 Retrieved 5 chunks


In [0]:
# Customize this question based on your PDF content
question2 = "What is the process for handling damaged goods during delivery?"

print(f"🔍 QUESTION: {question2}\n")
print("=" * 80)

result2 = generate_grounded_answer(question2)

print(f"\n📝 ANSWER:\n{result2['answer']}\n")
print("=" * 80)
print(f"\n📚 SOURCES:")
for doc, page in result2['sources']:
    print(f"  • {doc} (Page {page})")

print(f"\n🔢 Retrieved {len(result2['chunks_used'])} chunks")

🔍 QUESTION: What is the process for handling damaged goods during delivery?

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

📝 ANSWER:
According to Chunk 2, if there is damage to the goods, or any other product issues, then the supplier (and haulier if not managed or owned by the supplier) should be notified so that they can make adjustments to their invoice before sending to the customer. Additionally, the receiver can use the Receiving Advice message to notify the supplier that the goods have been received and confirm receipt of all products or only those where there may be a problem, such as damaged or missing goods.


📚 SOURCES:
  • UPS_Packaging_Guidelines.pdf (Page 8.0)
  • UPS_Packaging_Guidelines.pdf (Page 3.0)
  • UPS_Packaging_Guidelines.pdf (Page 1.0)
  • GS1_Logistic_Label_Guideline.pdf (Page 38.0)

🔢 Retrieved 5 

In [0]:
# This should trigger "I don't have that information" response
question3 = "What is the weather forecast for next week?"

print(f"🔍 QUESTION: {question3}\n")
print("=" * 80)

result3 = generate_grounded_answer(question3)

print(f"\n📝 ANSWER:\n{result3['answer']}\n")
print("=" * 80)
print(f"\n📚 SOURCES:")
for doc, page in result3['sources']:
    print(f"  • {doc} (Page {page})")

print(f"\n🔢 Retrieved {len(result3['chunks_used'])} chunks")
print("\n⚠️  Expected: Model should say it doesn't have this information")

🔍 QUESTION: What is the weather forecast for next week?

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

📝 ANSWER:
I don't have that information in the available documents.


📚 SOURCES:
  • arXiv_LLM_Network_Optimization_Supply_Chain.pdf (Page 14.0)
  • arXiv_Inventory_Management_ML.pdf (Page 6.0)
  • arXiv_LLM_Network_Optimization_Supply_Chain.pdf (Page 13.0)
  • arXiv_ML_Supply_Chain_Optimization.pdf (Page 14.0)
  • arXiv_Warehouse_Automation_Robotics.pdf (Page 10.0)

🔢 Retrieved 5 chunks

⚠️  Expected: Model should say it doesn't have this information


## Example Questions to Test

Here are 20 example questions you can try with this RAG system, organized by category:

### 📦 Packaging & Shipping Guidelines

1. "How do I calculate freight density for LTL shipments?"
2. "What are the requirements for stacking pallets safely?"
3. "What information should be included on a Bill of Lading?"
4. "How should I secure items on a pallet to prevent damage?"
5. "What are the guidelines for shrinkwrapping pallets?"
6. "What types of pallets should I use for international shipments?"

### 🏷️ Labeling & Standards (GS1)

7. "What is the GS1 logistic label standard?"
8. "How do I implement traceability in my supply chain?"
9. "What are the key elements of a traceability system?"
10. "What barcoding standards should I use for logistics?"

### 🤖 AI/ML in Supply Chain

11. "How can machine learning improve demand forecasting?"
12. "What are the applications of LLMs in supply chain optimization?"
13. "How can AI be used for last-mile delivery optimization?"
14. "What role does warehouse automation play in modern logistics?"
15. "How can generative AI improve supply chain resilience?"

### 🔍 Mixed/Complex Questions

16. "What are the main causes of freight damage during transit?"
17. "How do I handle temperature-sensitive shipments?"
18. "What are the best practices for reducing supply chain costs?"
19. "How can I improve inventory management using predictive analytics?"
20. "What safety protocols should be followed for hazardous materials shipping?"

### 🚫 Out-of-Scope Questions (Testing Boundaries)

* "What is the capital of France?" *(Should say "I don't have that information")*
* "How do I cook a pizza?" *(Should decline gracefully)*
* "What are the latest stock market trends?" *(Out of scope)*

---

**Note:** These questions are based on the types of documents in your knowledge base (shipping guides, GS1 standards, and AI/ML supply chain research papers). Copy any question into Cell 19 below to test!

In [0]:
# 🎯 Ask your own question here!
# Simply modify the question below and run this cell

# ============================================================================
# MODIFY THIS QUESTION:
my_question = "What information should be included on a Bill of Lading?"
# ============================================================================

# Some other questions you could try:
# - "How do I calculate freight density?"
# - "What are the requirements for international shipments?"
# - "What should I do if my shipment is damaged?"
# - "How do I properly label packages?"
# - "What are the guidelines for hazardous materials?"

print(f"🔍 YOUR QUESTION: {my_question}\n")
print("=" * 80)

my_result = generate_grounded_answer(my_question)

# Show all retrieved chunks (what the LLM had available)
print(f"\n📦 ALL RETRIEVED CHUNKS ({len(my_result['chunks_used'])} total):\n")
for i, chunk in enumerate(my_result['chunks_used'], 1):
    print(f"[Chunk {i}] {chunk['source_doc']} (Page {chunk['page']})")
    if chunk.get('score'):
        print(f"  Similarity Score: {chunk['score']:.4f}")
    print(f"  Text Preview: {chunk['chunk_text'][:200]}...\n")

print("=" * 80)

print(f"\n📝 ANSWER (from LLM):\n{my_result['answer']}\n")
print("=" * 80)
print(f"\n📚 SOURCES CITED IN ANSWER:")
for doc, page in my_result['sources']:
    print(f"  • {doc} (Page {page})")

print(f"\n💡 Notice: The LLM retrieved {len(my_result['chunks_used'])} chunks but may only cite the most relevant ones in the answer.")
print(f"💡 Tip: Edit 'my_question' above and re-run this cell to test another question!")

🔍 YOUR QUESTION: What information should be included on a Bill of Lading?

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

📦 ALL RETRIEVED CHUNKS (5 total):

[Chunk 1] GS1_Logistic_Label_Guideline.pdf (Page 21.0)
  Similarity Score: 0.6197
  Text Preview: GS1 Logistic Label Guideline 
Release 1.3, Ratified, Jul 2019 
© 2019 GS1 AISBL  
Page 21 of 58 
[5-1] The ship to information must relate to the physical address where the goods need to be 
delivered...

[Chunk 2] GS1_Logistic_Label_Guideline.pdf (Page 20.0)
  Similarity Score: 0.6131
  Text Preview: GS1 Logistic Label Guideline 
Release 1.3, Ratified, Jul 2019 
© 2019 GS1 AISBL  
Page 20 of 58 
5 
How to include transport and customer information 
5.1 
When would I use this? 
Carriers (Logistic s...

[Chunk 3] FedEx_Freight_Shipping_Guide.pdf (Page 2.0)
  Similarity Score:

## Next Steps

### Before Running

1. **Upload PDFs**: Add logistics documents to the Volume at `/Volumes/logistics_rag/knowledge_base/pdf_documents`
2. **Verify Endpoint Names**: 
   - Go to **Serving** → **Foundation Model APIs**
   - Confirm the embedding endpoint name (Section 3)
   - Confirm the LLM endpoint name (Section 5)
3. **Run Sections in Order**: Execute 1 → 2 → 3 → 4 → 5 → 6

### Customization Ideas

* **Chunking**: Adjust `chunk_size` and `overlap` in Section 2
* **Retrieval**: Modify `k` (number of chunks) in Sections 4-6
* **Prompt**: Edit the system instructions in Section 5 for different behavior
* **Metadata**: Add fields like `doc_date`, `doc_type` to the Delta table for filtered retrieval
* **Reranking**: Add a reranker model between retrieval and generation
* **Continuous Sync**: Change `pipeline_type="CONTINUOUS"` in Section 3 for real-time updates

### Productionization

* **Model Serving**: Wrap the `generate_grounded_answer` function in a Model Serving endpoint
* **Evaluation**: Use MLflow to track question/answer quality
* **Monitoring**: Log queries, latency, and chunk relevance scores
* **Access Control**: Use UC permissions to restrict which users can query which documents

---

**You now have a complete RAG pipeline using only Databricks-native components!**